# TripoSR · Kaggle 双 T4 常驻 Worker

这份 Notebook 专门用于 Kaggle：Notebook 内核可以保持 **Python 3.12**，TripoSR 始终通过 `uv` 创建的 **Python 3.10 `.venv`** 运行。

设计目标：

- TripoSR **每张 T4 常驻一份模型**，不再每个任务重新执行 `run.py` / 重载 1.68 GB 权重。
- `rembg[gpu]` + `onnxruntime-gpu`，并把两个 rembg Session 分别固定到 GPU0 / GPU1。
- `torchmcubes` 直接安装 `kaggle-build` Release 的 `sm_75` wheel，**不现场 CMake/NVCC 编译**。
- 两个 GPU 进程各自通过不可缓存的 `POST /task/claim` 领取 `triposr` 任务，天然双卡并行。
- 输入只在内存中处理，结果 GLB/OBJ 经过 AES-GCM 加密后上传回 Hub。
- 模型下载和初始化只发生在 Worker 启动阶段；后续单张任务只承担 rembg + TripoSR + mesh + 上传。
- 启动前强制校验 Hub 协议、Token、`triposr` 路由和 SQLite 共享队列实例，版本不匹配时直接停止。

Kaggle 要求：**GPU T4 x2 + Internet**。


## Cell 1 · 安装 Python 3.10 TripoSR Runtime

In [1]:
import os
import shutil
import subprocess
from pathlib import Path

os.environ["UV_LINK_MODE"] = "copy"
os.environ.pop("UV_SYSTEM_PYTHON", None)
os.environ["IPYTHONDIR"] = "/kaggle/working/.ipython"

ROOT = Path("/kaggle/working/TripoSR")
UV = "/usr/local/bin/uv"
PY = ROOT / ".venv/bin/python"
REPO = "https://github.com/VAST-AI-Research/TripoSR.git"
WHEEL_URL = (
    "https://github.com/xiaoqianran/kaggle-build/releases/download/"
    "triposr-py310-torch2.7.1-cu128-sm75/"
    "torchmcubes-0.1.0-cp310-cp310-linux_x86_64.whl"
)

def run(cmd, cwd=None):
    print("\n>>>", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True, cwd=str(cwd) if cwd else None)

if ROOT.exists():
    shutil.rmtree(ROOT)

run(["git", "clone", "--depth", "1", REPO, ROOT])
run([UV, "venv", "--no-managed-python", "--python", "/usr/bin/python3.10", ROOT / ".venv"])
run([
    UV, "pip", "install", "--python", PY,
    "torch==2.7.1", "torchvision==0.22.1",
    "--index-url", "https://download.pytorch.org/whl/cu128",
])

req = ROOT / "requirements.txt"
req_fixed = ROOT / "requirements.kaggle.txt"
req_fixed.write_text(
    "\n".join(line for line in req.read_text().splitlines() if "torchmcubes" not in line.lower()) + "\n"
)
run([UV, "pip", "install", "--python", PY, "-r", req_fixed])
run([UV, "pip", "install", "--python", PY, "rembg[gpu]", "cryptography", "requests"])
run([UV, "pip", "install", "--python", PY, WHEEL_URL])

check = r"""
import torch, torchmcubes, onnxruntime as ort
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU{i}:", torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
print("ORT:", ort.__version__)
print("ORT providers:", ort.get_available_providers())
assert torch.cuda.is_available()
assert "CUDAExecutionProvider" in ort.get_available_providers()
x=torch.randn(16,16,16,device="cuda:0")
v,f=torchmcubes.marching_cubes(x,0.0)
print("torchmcubes CUDA OK:", v.shape, f.shape)
"""
run([PY, "-c", check])
print("\n✅ Python 3.10 TripoSR runtime ready:", PY)


>>> git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR.git /kaggle/working/TripoSR


Cloning into '/kaggle/working/TripoSR'...



>>> /usr/local/bin/uv venv --no-managed-python --python /usr/bin/python3.10 /kaggle/working/TripoSR/.venv


Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: TripoSR/.venv
Activate with: source TripoSR/.venv/bin/activate
Using Python 3.10.12 environment at: TripoSR/.venv



>>> /usr/local/bin/uv pip install --python /kaggle/working/TripoSR/.venv/bin/python torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu128


...


>>> /usr/local/bin/uv pip install --python /kaggle/working/TripoSR/.venv/bin/python rembg[gpu] cryptography requests


Resolved 38 packages in 424ms
Prepared 8 packages in 3.09s
Installed 8 packages in 348ms
 + cffi==2.1.1
 + coloredlogs==15.0.1
 + cryptography==50.0.0
 + flatbuffers==25.12.19
 + humanfriendly==10.0
 + onnxruntime-gpu==1.23.2
 + protobuf==7.35.1
 + pycparser==3.0
Using Python 3.10.12 environment at: TripoSR/.venv



>>> /usr/local/bin/uv pip install --python /kaggle/working/TripoSR/.venv/bin/python https://github.com/xiaoqianran/kaggle-build/releases/download/triposr-py310-torch2.7.1-cu128-sm75/torchmcubes-0.1.0-cp310-cp310-linux_x86_64.whl


Resolved 27 packages in 445ms
Prepared 1 package in 22ms
Installed 1 package in 2ms
 + torchmcubes==0.1.0 (from https://github.com/xiaoqianran/kaggle-build/releases/download/triposr-py310-torch2.7.1-cu128-sm75/torchmcubes-0.1.0-cp310-cp310-linux_x86_64.whl)



>>> /kaggle/working/TripoSR/.venv/bin/python -c 
import torch, torchmcubes, onnxruntime as ort
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU{i}:", torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
print("ORT:", ort.__version__)
print("ORT providers:", ort.get_available_providers())
assert torch.cuda.is_available()
assert "CUDAExecutionProvider" in ort.get_available_providers()
x=torch.randn(16,16,16,device="cuda:0")
v,f=torchmcubes.marching_cubes(x,0.0)
print("torchmcubes CUDA OK:", v.shape, f.shape)

Torch: 2.7.1+cu128 CUDA: 12.8
GPU count: 2
GPU0: Tesla T4 (7, 5)
GPU1: Tesla T4 (7, 5)
ORT: 1.23.2
ORT providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
torchmcubes CUDA OK: torch.Size([31227, 3]) torch.Size([10409, 3])

✅ Python 3.10 TripoSR runtime ready: /kaggle/working/TripoSR/.venv/bin/python


## Cell 2 · 写入常驻双 GPU Worker

Notebook 的 Python 3.12 **不直接 import TripoSR**。这一格只把 Worker 写成 `.py` 文件，后面始终使用 `.venv/bin/python` 启动。

In [2]:
from pathlib import Path

ROOT = Path("/kaggle/working/TripoSR")
WORKER = ROOT / "kaggle_worker.py"

WORKER_SOURCE = r'''from __future__ import annotations

import hashlib
import io
import multiprocessing as mp
import os
import signal
import socket
import threading
import time
import traceback
import uuid
from typing import Any
from urllib.parse import urljoin

import numpy as np
import rembg
import requests
import torch
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from PIL import Image
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground

BASE_URL = os.environ["BASE_URL"].rstrip("/") + "/"
TOKEN = os.environ["KAGGLE_HUB_TOKEN"]
MODEL_ID = os.getenv("TRIPOSR_MODEL_ID", "stabilityai/TripoSR")
REMBG_MODEL = os.getenv("TRIPOSR_REMBG_MODEL", "u2net")
POLL_TIMEOUT = 35
REQUEST_TIMEOUT = 120
HEARTBEAT_SECONDS = 10
IDLE_DIAGNOSTIC_SECONDS = 60


def encrypt_blob(data: bytes) -> bytes:
    key = hashlib.sha256(TOKEN.encode()).digest()
    nonce = os.urandom(12)
    return nonce + AESGCM(key).encrypt(nonce, data, None)


def api_url(path: str) -> str:
    return urljoin(BASE_URL, path.lstrip("/"))


def auth_headers() -> dict[str, str]:
    return {
        "Authorization": f"Bearer {TOKEN}",
        "Cache-Control": "no-cache, no-store",
        "Pragma": "no-cache",
    }


def checked_response(response: requests.Response, label: str) -> requests.Response:
    if response.status_code >= 400:
        body = response.text[:1000].replace("\n", " ")
        raise RuntimeError(f"{label}: HTTP {response.status_code} | {body}")
    return response


def server_snapshot(session: requests.Session) -> dict[str, Any]:
    response = session.get(api_url("/api/status"), params={"_ts": time.time_ns()}, timeout=15)
    checked_response(response, "GET /api/status")
    return response.json()


def preflight_hub() -> str:
    """Verify URL, protocol, TripoSR route, token, and shared SQLite state."""
    session = requests.Session()
    session.headers.update(auth_headers())
    response = session.get(api_url("/api/models"), params={"_ts": time.time_ns()}, timeout=20)
    checked_response(response, "GET /api/models")
    model_ids = {item.get("id") for item in response.json() if isinstance(item, dict)}
    if "triposr" not in model_ids:
        raise RuntimeError(f"Hub does not advertise triposr. Models={sorted(x for x in model_ids if x)}")

    # Authenticated read verifies the Bearer token without claiming a task.
    checked_response(
        session.get(api_url("/api/failed"), params={"_ts": time.time_ns()}, timeout=20),
        "GET /api/failed (auth check)",
    )
    snapshot = server_snapshot(session)
    if snapshot.get("storage") != "sqlite":
        raise RuntimeError(
            "Connected Hub is an old in-memory build. Update/restart the local Hub before starting 003."
        )
    instance_id = str(snapshot.get("hub_instance_id") or "")
    if not instance_id:
        raise RuntimeError("Hub did not return hub_instance_id; local Hub and 003 protocol versions differ")
    queued = int(snapshot.get("queued_by_model", {}).get("triposr", 0) or 0)
    inflight = int(snapshot.get("inflight_by_model", {}).get("triposr", 0) or 0)
    print(
        f"[preflight] Hub OK | instance={instance_id[:12]} | storage=sqlite | "
        f"triposr queued={queued} inflight={inflight}",
        flush=True,
    )
    return instance_id


def prefetch_model() -> None:
    from huggingface_hub import snapshot_download

    print(f"[prefetch] {MODEL_ID}", flush=True)
    snapshot_download(repo_id=MODEL_ID, allow_patterns=["config.yaml", "model.ckpt"])
    print("[prefetch] TripoSR assets ready", flush=True)


def build_rembg_session(gpu: int):
    providers = [
        ("CUDAExecutionProvider", {"device_id": gpu}),
        "CPUExecutionProvider",
    ]
    session = rembg.new_session(REMBG_MODEL, providers=providers)
    active = session.inner_session.get_providers()
    options = session.inner_session.get_provider_options()
    if "CUDAExecutionProvider" not in active:
        raise RuntimeError(f"rembg CUDA provider unavailable on GPU{gpu}: {active}")
    print(
        f"[GPU{gpu}] rembg={REMBG_MODEL} providers={active} options={options.get('CUDAExecutionProvider', {})}",
        flush=True,
    )
    return session


def prepare_image(raw: bytes, session, remove_bg: bool, foreground_ratio: float) -> Image.Image:
    image = Image.open(io.BytesIO(raw))
    if not remove_bg:
        return image.convert("RGB")

    image = remove_background(image, session)
    image = resize_foreground(image, foreground_ratio)
    arr = np.asarray(image).astype(np.float32) / 255.0
    arr = arr[:, :, :3] * arr[:, :, 3:4] + (1.0 - arr[:, :, 3:4]) * 0.5
    return Image.fromarray((arr * 255.0).astype(np.uint8))


def export_mesh(mesh, output_format: str) -> bytes:
    data = mesh.export(file_type=output_format)
    if isinstance(data, str):
        return data.encode("utf-8")
    return bytes(data)


def heartbeat_loop(
    worker_id: str,
    gpu: int,
    stop: threading.Event,
    active_task: dict[str, int | None],
) -> None:
    session = requests.Session()
    session.headers.update(auth_headers())
    while not stop.wait(HEARTBEAT_SECONDS):
        try:
            session.post(
                api_url("/worker/heartbeat"),
                json={
                    "worker_id": worker_id,
                    "local_queue": 0,
                    "upload_queue": 0,
                    "active_task_id": active_task["id"],
                    "meta": {"gpu_index": gpu, "persistent": True},
                },
                timeout=15,
            ).raise_for_status()
        except Exception as exc:
            print(f"[GPU{gpu}] heartbeat: {type(exc).__name__}: {exc}", flush=True)


def report_failure(session: requests.Session, task_id: int, exc: BaseException, gpu: int) -> None:
    message = f"{type(exc).__name__}: {exc}"
    print(f"[GPU{gpu}] FAIL #{task_id}: {message}", flush=True)
    try:
        session.post(
            api_url("/task/fail"),
            json={"id": task_id, "error": message[:1900], "requeue": True},
            timeout=20,
        ).raise_for_status()
    except Exception as report_exc:
        print(f"[GPU{gpu}] fail-report error: {report_exc}", flush=True)


def gpu_worker(gpu: int, run_id: str, hub_instance_id: str) -> None:
    torch.cuda.set_device(gpu)
    device = f"cuda:{gpu}"
    gpu_name = torch.cuda.get_device_name(gpu)
    worker_id = f"triposr-{run_id}-g{gpu}"

    print(f"[GPU{gpu}] loading TripoSR on {gpu_name} ...", flush=True)
    started = time.perf_counter()
    model = TSR.from_pretrained(MODEL_ID, config_name="config.yaml", weight_name="model.ckpt")
    model.renderer.set_chunk_size(8192)
    model.to(device)
    model.eval()
    rembg_session = build_rembg_session(gpu)
    print(f"[GPU{gpu}] READY in {time.perf_counter()-started:.2f}s", flush=True)

    session = requests.Session()
    session.headers.update(auth_headers())
    register_response = session.post(
        api_url("/worker/register"),
        json={
            "worker_id": worker_id,
            "model": "triposr",
            "gpus": [gpu_name],
            "runtime": "triposr-persistent-py310",
            "concurrency": 1,
            "meta": {
                "gpu_index": gpu,
                "torch": torch.__version__,
                "torch_cuda": torch.version.cuda,
                "rembg_model": REMBG_MODEL,
                "rembg_providers": rembg_session.inner_session.get_providers(),
                "persistent": True,
            },
        },
        timeout=30,
    )
    checked_response(register_response, "POST /worker/register")
    print(f"[GPU{gpu}] registered as {worker_id}", flush=True)

    stop = threading.Event()
    active_task: dict[str, int | None] = {"id": None}
    heartbeat = threading.Thread(
        target=heartbeat_loop,
        args=(worker_id, gpu, stop, active_task),
        daemon=True,
    )
    heartbeat.start()

    def shutdown(*_args):
        stop.set()
        raise KeyboardInterrupt

    signal.signal(signal.SIGTERM, shutdown)

    last_idle_diagnostic = 0.0
    try:
        while True:
            task: dict[str, Any] | None = None
            try:
                # POST cannot be cached as a stale 204 by a tunnel/CDN.
                response = session.post(
                    api_url("/task/claim"),
                    json={"model": "triposr", "worker_id": worker_id, "wait_seconds": 25},
                    timeout=POLL_TIMEOUT,
                )
                response_instance = response.headers.get("X-Hub-Instance", "")
                if response_instance and response_instance != hub_instance_id:
                    raise RuntimeError(
                        f"Hub instance changed: expected={hub_instance_id[:12]} got={response_instance[:12]}. "
                        "Tunnel may point at multiple unrelated Hub databases."
                    )
                if response.status_code == 204:
                    now = time.monotonic()
                    if now - last_idle_diagnostic >= IDLE_DIAGNOSTIC_SECONDS:
                        last_idle_diagnostic = now
                        snapshot = server_snapshot(session)
                        queued = int(snapshot.get("queued_by_model", {}).get("triposr", 0) or 0)
                        inflight = int(snapshot.get("inflight_by_model", {}).get("triposr", 0) or 0)
                        print(
                            f"[GPU{gpu}] idle | Hub triposr queued={queued} inflight={inflight} "
                            f"instance={str(snapshot.get('hub_instance_id', ''))[:12]}",
                            flush=True,
                        )
                    continue
                checked_response(response, "POST /task/claim")
                task = response.json()
                task_id = int(task["id"])
                active_task["id"] = task_id
                print(
                    f"[GPU{gpu}] ↓ #{task_id} {task.get('source_label','input')} "
                    f"res={task.get('mc_resolution',256)} fmt={task.get('output_format','glb')}",
                    flush=True,
                )

                t0 = time.perf_counter()
                input_response = session.get(api_url(task["input_url"]), timeout=60)
                input_response.raise_for_status()
                t_download = time.perf_counter()

                image = prepare_image(
                    input_response.content,
                    rembg_session,
                    bool(task.get("remove_background", True)),
                    float(task.get("foreground_ratio", 0.85)),
                )
                t_pre = time.perf_counter()

                model.renderer.set_chunk_size(int(task.get("chunk_size", 8192)))
                with torch.inference_mode():
                    scene_codes = model([image], device=device)
                t_model = time.perf_counter()

                with torch.inference_mode():
                    meshes = model.extract_mesh(
                        scene_codes,
                        True,
                        resolution=int(task.get("mc_resolution", 256)),
                    )
                t_mesh = time.perf_counter()

                mesh = meshes[0]
                output_format = str(task.get("output_format", "glb")).lower()
                artifact = export_mesh(mesh, output_format)
                encrypted = encrypt_blob(artifact)
                t_export = time.perf_counter()

                elapsed = t_export - t0
                upload = session.post(
                    api_url("/upload/artifact"),
                    data={
                        "id": str(task_id),
                        "model": "triposr",
                        "worker_id": worker_id,
                        "gpu": str(gpu),
                        "seconds": f"{elapsed:.3f}",
                        "output_format": output_format,
                        "vertices": str(len(mesh.vertices)),
                        "faces": str(len(mesh.faces)),
                    },
                    files={
                        "file": (
                            f"{task_id}.{output_format}.bin",
                            encrypted,
                            "application/octet-stream",
                        )
                    },
                    timeout=REQUEST_TIMEOUT,
                )
                upload.raise_for_status()
                t_up = time.perf_counter()
                active_task["id"] = None

                print(
                    f"[GPU{gpu}] ✓ #{task_id} total={elapsed:.2f}s "
                    f"download={t_download-t0:.2f}s rembg={t_pre-t_download:.2f}s "
                    f"model={t_model-t_pre:.2f}s mesh={t_mesh-t_model:.2f}s "
                    f"export={t_export-t_mesh:.2f}s upload={t_up-t_export:.2f}s "
                    f"v={len(mesh.vertices)} f={len(mesh.faces)}",
                    flush=True,
                )

                del image, scene_codes, meshes, mesh, artifact, encrypted
                torch.cuda.empty_cache()

            except KeyboardInterrupt:
                raise
            except Exception as exc:
                if task is not None and "id" in task:
                    report_failure(session, int(task["id"]), exc, gpu)
                    active_task["id"] = None
                else:
                    print(f"[GPU{gpu}] poll error: {type(exc).__name__}: {exc}", flush=True)
                    time.sleep(2)
                traceback.print_exc()
    except KeyboardInterrupt:
        pass
    finally:
        stop.set()
        print(f"[GPU{gpu}] stopped", flush=True)


def main() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available")
    gpu_count = torch.cuda.device_count()
    if gpu_count < 1:
        raise RuntimeError("No CUDA GPU found")

    wanted = int(os.getenv("TRIPOSR_GPU_COUNT", str(gpu_count)))
    gpu_count = min(gpu_count, max(1, wanted))
    run_id = f"{socket.gethostname()[:8]}-{uuid.uuid4().hex[:6]}"

    print(
        f"TripoSR persistent worker | GPUs={gpu_count} | rembg={REMBG_MODEL} | base={BASE_URL}",
        flush=True,
    )
    hub_instance_id = preflight_hub()
    prefetch_model()

    ctx = mp.get_context("spawn")
    processes = [
        ctx.Process(target=gpu_worker, args=(gpu, run_id, hub_instance_id), name=f"triposr-gpu{gpu}")
        for gpu in range(gpu_count)
    ]
    for process in processes:
        process.start()

    try:
        for process in processes:
            process.join()
    except KeyboardInterrupt:
        print("Stopping workers ...", flush=True)
        for process in processes:
            if process.is_alive():
                process.terminate()
        for process in processes:
            process.join(timeout=10)


if __name__ == "__main__":
    mp.freeze_support()
    main()
'''
WORKER.write_text(WORKER_SOURCE)
print("✅ Worker written:", WORKER)
print("Lines:", len(WORKER_SOURCE.splitlines()))


✅ Worker written: /kaggle/working/TripoSR/kaggle_worker.py
Lines: 402


## Cell 3 · 启动双 T4 常驻 Worker

003 已固定使用：

- Hub：`https://ranran-sana.202820.xyz`
- Bearer / AES-GCM：`wangran`
- rembg：`u2net` + CUDAExecutionProvider

启动时先检查 Hub、`triposr` 路由、Token、Hub 实例 ID 和 SQLite 状态层。Worker 使用不可缓存的 `POST /task/claim`，并在心跳中持续续租正在推理的任务。若本地仍是旧版内存队列，003 会拒绝启动并明确提示先更新 Hub。


In [3]:
import os
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/working/TripoSR")
PY = ROOT / ".venv/bin/python"
WORKER = ROOT / "kaggle_worker.py"
LOG = Path("/kaggle/working/triposr-worker.log")
CLAIM_LOG = Path("/kaggle/working/triposr-claimed-tasks.jsonl")
PID_FILE = Path("/kaggle/working/triposr-worker.pid")

DEFAULT_BASE_URL = "https://ranran-sana.202820.xyz"
DEFAULT_TOKEN = "wangran"

# 003 固定使用这个 Hub。若以后域名/Token 改了，直接改下面两行。
# 不再从旧环境变量、PASSWORD 或 Kaggle Secrets 静默覆盖。
BASE_URL = DEFAULT_BASE_URL.rstrip("/")
TOKEN = DEFAULT_TOKEN

REMBG_MODEL = "u2net"
GPU_COUNT = 2

if PID_FILE.exists():
    try:
        old_pid = int(PID_FILE.read_text().strip())
        os.kill(old_pid, 0)
        raise RuntimeError(f"Worker 已在运行 PID={old_pid}；先执行停止 Cell")
    except ProcessLookupError:
        PID_FILE.unlink(missing_ok=True)

LOG.write_text("")
CLAIM_LOG.write_text("")
env = os.environ.copy()
env.update({
    "BASE_URL": BASE_URL,
    "KAGGLE_HUB_TOKEN": TOKEN,
    "TRIPOSR_REMBG_MODEL": REMBG_MODEL,
    "TRIPOSR_GPU_COUNT": str(GPU_COUNT),
    "TRIPOSR_CLAIM_LOG": str(CLAIM_LOG),
    "HF_HOME": "/kaggle/working/hf-cache",
    "U2NET_HOME": "/kaggle/working/.u2net",
    "PYTHONUNBUFFERED": "1",
})

log_handle = LOG.open("ab", buffering=0)
proc = subprocess.Popen(
    [str(PY), str(WORKER)],
    cwd=str(ROOT),
    env=env,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
PID_FILE.write_text(str(proc.pid))

print("✅ TripoSR persistent worker started")
print("PID:", proc.pid)
print("Hub:", BASE_URL)
print("Token: configured (value hidden)")
print("Log:", LOG)
print("Claim records:", CLAIM_LOG)
print("前几十秒会下载/加载模型；之后两个 GPU 常驻，不再按任务重新初始化。")


✅ TripoSR persistent worker started
PID: 255
Hub: https://ranran-sana.202820.xyz
Token: configured (value hidden)
Log: /kaggle/working/triposr-worker.log
前几十秒会下载/加载模型；之后两个 GPU 常驻，不再按任务重新初始化。


## Cell 4 · 查看 Worker / Hub / 队列状态

这格会同时显示 Kaggle 看到的 `triposr queued / inflight` 与 Worker 日志。

新版 Hub 使用 SQLite 共享队列，多 Uvicorn 进程和 Hub 重启都不会再拆散或清空任务。这里同时显示 `hub_instance_id` 与存储类型；若实例 ID 在请求间变化，说明 Tunnel 后面连接了不同的 Hub 数据库。

此外会读取 `triposr-claimed-tasks.jsonl`，显示最近 20 条 Worker 实际领取到的任务记录（完整 task JSON，敏感字段自动脱敏）。


In [4]:
import json
import os
import subprocess
import time
from pathlib import Path
from urllib.request import Request, urlopen

PID_FILE = Path("/kaggle/working/triposr-worker.pid")
LOG = Path("/kaggle/working/triposr-worker.log")
CLAIM_LOG = Path("/kaggle/working/triposr-claimed-tasks.jsonl")
BASE_URL = "https://ranran-sana.202820.xyz"
TOKEN = "wangran"

if PID_FILE.exists():
    pid = int(PID_FILE.read_text().strip())
    try:
        os.kill(pid, 0)
        print("Worker: RUNNING | PID", pid)
    except ProcessLookupError:
        print("Worker: EXITED | PID", pid)
else:
    print("Worker: NOT STARTED")

print("\n=== HUB (seen from Kaggle) ===")
try:
    req = Request(
        f"{BASE_URL}/api/status?_ts={time.time_ns()}",
        headers={
            "Authorization": f"Bearer {TOKEN}",
            "Cache-Control": "no-cache, no-store",
            "Pragma": "no-cache",
        },
    )
    with urlopen(req, timeout=15) as r:
        status = json.loads(r.read().decode("utf-8"))
    print("hub instance    :", status.get("hub_instance_id", "missing"))
    print("state storage   :", status.get("storage", "old/in-memory"))
    print("server process  :", status.get("process_id", "unknown"))
    print("triposr queued  :", status.get("queued_by_model", {}).get("triposr", 0))
    print("triposr inflight:", status.get("inflight_by_model", {}).get("triposr", 0))
    online = [w for w in status.get("workers", []) if w.get("online") and w.get("model") == "triposr"]
    print("online TripoSR workers:", len(online))
    for w in online:
        print(" -", w.get("worker_id"), "|", ", ".join(w.get("gpus") or []), "| active", w.get("active_task_id"))
except Exception as e:
    print("Hub check FAILED:", type(e).__name__, e)



print("\n=== Recently claimed tasks ===")
if not CLAIM_LOG.exists() or CLAIM_LOG.stat().st_size == 0:
    print("No claimed task records yet.")
else:
    raw_lines = CLAIM_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]
    for raw in raw_lines:
        try:
            record = json.loads(raw)
            task = record.get("task") or {}
            print(
                f"\n[{record.get('received_at', '?')}] "
                f"GPU{record.get('gpu', '?')} | worker={record.get('worker_id', '?')} | "
                f"task_id={task.get('id', '?')}"
            )
            print(json.dumps(task, ensure_ascii=False, indent=2, default=str))
        except Exception:
            print(raw)

print("\n=== GPU ===")
subprocess.run(["nvidia-smi", "--query-gpu=index,name,utilization.gpu,memory.used,memory.total", "--format=csv"])
print("\n=== Last 140 log lines ===")
if LOG.exists():
    subprocess.run(["tail", "-n", "140", str(LOG)])


Worker: RUNNING | PID 255

=== HUB (seen from Kaggle) ===
Hub check FAILED: HTTPError HTTP Error 403: Forbidden

=== GPU ===
index, name, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
0, Tesla T4, 0 %, 0 MiB, 15360 MiB
1, Tesla T4, 0 %, 0 MiB, 15360 MiB

=== Last 140 log lines ===


## Cell 5 · 停止 Worker

In [5]:
# import os
# import signal
# import time
# from pathlib import Path

# PID_FILE = Path("/kaggle/working/triposr-worker.pid")
# if not PID_FILE.exists():
#     print("没有运行中的 Worker")
# else:
#     pid = int(PID_FILE.read_text().strip())
#     try:
#         os.killpg(pid, signal.SIGTERM)
#         print("Stopping process group:", pid)
#         time.sleep(2)
#     except ProcessLookupError:
#         pass
#     PID_FILE.unlink(missing_ok=True)
#     print("✅ stopped")

## 运行方式

1. 更新本地源码后重启 Hub：`uv run python recv.py`。新版 SQLite 队列也支持 `uvicorn recv:app --workers 4`。
2. Cloudflare Tunnel 指向 `http://localhost:30100`；003 固定连接 `https://ranran-sana.202820.xyz`。
3. Kaggle 依次执行 Cell 1 → 2 → 3。Token 已内置为 `wangran`，无需 Kaggle Secret。
4. 执行 Cell 4，应看到两个 TripoSR Worker 在线。
5. 在本地 UI 上传图片或点击“转为 3D”。日志出现 `CLAIMED_TASK` / `↓ #ID` 就说明读取队列正常；Cell 4 会直接显示最近领取到的完整任务记录。

### 判断问题在哪

- `401`：本地 Hub 的 Token 不是 `wangran`。
- 启动时报 `old in-memory build`：Cloudflare 指向的 Hub 还没更新或没重启。
- `404`：Cloudflare 地址指向错误 Hub，或本地没有新版 `/task/claim`。
- `502/503` 或连接失败：Tunnel / 本地 Hub 没通。
- `hub_instance_id` 发生变化：Tunnel 后面存在多个未共享同一 SQLite 文件的 Hub。
- `storage` 不是 `sqlite`：公网仍连着旧版 Hub。
- Kaggle 看到 `queued=0`，但本地 UI 显示已入队：UI 与 Cloudflare 没连到同一个 Hub 实例/端口。
- 出现 `↓ #ID` 后才报错：队列没问题，继续看图片下载、rembg、TripoSR 推理或上传阶段日志。
